# 61 - Position Sizing Study

Compare different position sizing approaches for the Checkmate signal:

1. **Original** - Goes to 0% and negative (broken)
2. **Floor** - Minimum 20% allocation always
3. **Linear** - Smooth scaling 20-100%
4. **Asymmetric** - Aggressive entry, gradual exit
5. **Trigger** - Only act on extreme signals
6. **DCA Multiplier** - Adjust DCA amount based on signal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"

def load_metric(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists(): return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if df['time'].dt.tz is not None: df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    return df

data = {m: load_metric(m) for m in ['price', 'mvrv']}
print("Data loaded")

In [ ]:
# Position sizing functions
def position_original(mvrv):
    """Original (broken) - goes to 0% and negative"""
    if mvrv < 1.0:   return 1.00
    elif mvrv < 1.5: return 0.75
    elif mvrv < 2.0: return 0.50
    elif mvrv < 2.4: return 0.00  # ❌ Zero!
    elif mvrv < 3.0: return -0.25  # ❌ Short!
    else:            return -0.50  # ❌ More short!

def position_floor(mvrv, min_pos=0.20):
    """Simple fix - minimum floor"""
    if mvrv < 1.0:   return 1.00
    elif mvrv < 1.5: return 0.80
    elif mvrv < 2.0: return 0.60
    elif mvrv < 2.5: return 0.40
    elif mvrv < 3.0: return 0.30
    else:            return min_pos  # ✅ Never below floor

def position_linear(mvrv, min_pos=0.20, max_pos=1.0):
    """Linear scaling between min and max"""
    # MVRV range: typically 0.5 to 4.0
    normalized = (mvrv - 0.5) / (4.0 - 0.5)
    normalized = max(0, min(1, normalized))
    return max_pos - normalized * (max_pos - min_pos)

def position_asymmetric(mvrv):
    """Asymmetric - aggressive entry, gradual exit"""
    if mvrv < 0.8:   return 1.00  # Very aggressive
    elif mvrv < 1.0: return 1.00
    elif mvrv < 1.2: return 0.90
    elif mvrv < 1.5: return 0.80
    elif mvrv < 2.0: return 0.65
    elif mvrv < 2.5: return 0.50
    elif mvrv < 3.0: return 0.35
    else:            return 0.25  # Slow exit

# Build dataframe
df = data['price'][['value']].rename(columns={'value': 'price'})
df['returns'] = df['price'].pct_change()
df = df.join(data['mvrv'][['value']].rename(columns={'value': 'mvrv'}))
df['mvrv'] = df['mvrv'].ffill().fillna(1.0)

print("Position sizing functions defined")

In [ ]:
# Backtest each strategy
strategies = [
    ('Original (broken)', position_original),
    ('Floor (20%)', position_floor),
    ('Linear', position_linear),
    ('Asymmetric', position_asymmetric),
]

results = []
years = (df.index[-1] - df.index[0]).days / 365

# HODL baseline
hodl = 100000 * (1 + df['returns']).cumprod()
hodl_cagr = ((hodl.iloc[-1] / 100000) ** (1/years) - 1) * 100
hodl_dd = (hodl / hodl.cummax() - 1).min() * 100

for name, func in strategies:
    df['pos'] = df['mvrv'].apply(func).shift(1)
    df['strat_ret'] = df['pos'] * df['returns']
    df['equity'] = 100000 * (1 + df['strat_ret']).cumprod()
    df['dd'] = df['equity'] / df['equity'].cummax() - 1
    
    results.append({
        'name': name,
        'cagr': ((df['equity'].iloc[-1] / 100000) ** (1/years) - 1) * 100,
        'max_dd': df['dd'].min() * 100,
        'sharpe': (df['strat_ret'].mean() / df['strat_ret'].std()) * np.sqrt(365),
        'avg_pos': df['pos'].mean(),
        'equity': df['equity'].copy()
    })

# Print results
print(f"\n{'Strategy':<20} {'CAGR':>10} {'MaxDD':>10} {'Sharpe':>10} {'AvgPos':>10}")
print("-"*60)
print(f"{'HODL':<20} {hodl_cagr:>9.1f}% {hodl_dd:>9.1f}% {'--':>10} {'100%':>10}")
for r in results:
    print(f"{r['name']:<20} {r['cagr']:>9.1f}% {r['max_dd']:>9.1f}% {r['sharpe']:>10.2f} {r['avg_pos']:>9.0%}")

In [ ]:
# Visualize position sizing functions
mvrv_range = np.linspace(0.5, 4.0, 100)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#ef4444', '#22c55e', '#3b82f6', '#a855f7']

for (name, func), c in zip(strategies, colors):
    positions = [func(m) for m in mvrv_range]
    ax.plot(mvrv_range, [p*100 for p in positions], label=name, color=c, linewidth=2)

ax.axhline(y=0, color='white', linestyle='--', alpha=0.3)
ax.axhline(y=100, color='white', linestyle='--', alpha=0.3)
ax.set_xlabel('MVRV')
ax.set_ylabel('Position (%)')
ax.set_title('Position Sizing Functions Comparison', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusion

**Key Principle:** Use the signal for RISK ADJUSTMENT, not market timing.

**Recommended approach:** Asymmetric positioning
- Aggressive on entry (100% when bullish)
- Gradual on exit (never below 20-25%)
- Captures upside, reduces drawdowns